### NBA Player Similarity & Development Reports

This notebook and report are designed to compare players to similar player profiles, helping identify key strengths, development areas, and potential pathways for growth within their game.

## 1. Imports

In [8]:
import time
import warnings
from pathlib import Path
from io import BytesIO
import urllib.request
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
from matplotlib.patches import Arc, Circle, Rectangle as MplRectangle, Wedge, Polygon, PathPatch
from matplotlib.path import Path
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
import pathlib

from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import (
    leaguedashplayerstats,
    leaguedashplayershotlocations,
    commonplayerinfo
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

## 2. User Inputs

In [9]:
PLAYER_NAME = "Andrew Wiggins"
SEASON = "2015-16"
SEASON_TYPE = "Regular Season"

def build_season_strings(latest_season, num_seasons):
    start_year = int(latest_season.split("-")[0])
    return [
        f"{year}-{str(year + 1)[-2:]}"
        for year in range(start_year, start_year - num_seasons, -1)
    ]

COMP_SEASONS = build_season_strings("2025-26", 50)

PER_MODE = "Per100Possessions"
MIN_GP = 20
MIN_MINUTES_PER_GAME = 15

OUTPUT_DIR = pathlib.Path("player_development_reports")
OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Team Colors, Logos, & API Helpers

In [10]:
NBA_TEAMS = teams.get_teams()
TEAM_ABBR_LOOKUP = {t["abbreviation"]: t for t in NBA_TEAMS}

TEAM_COLORS = {
    "ATL": ("#E03A3E", "#C1D32F", "https://en.wikipedia.org/wiki/Special:FilePath/Atlanta_Hawks_logo.svg?width=512"),
    "BOS": ("#007A33", "#BA9653", "https://en.wikipedia.org/wiki/Special:FilePath/Boston_Celtics.svg?width=512"),
    "BKN": ("#000000", "#FFFFFF", "https://en.wikipedia.org/wiki/Special:FilePath/Brooklyn_Nets_newlogo.svg?width=512"),
    "CHA": ("#1D1160", "#00788C", "https://en.wikipedia.org/wiki/Special:FilePath/Charlotte_Hornets_(2014).svg?width=512"),
    "CHI": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Chicago_Bulls_logo.svg?width=512"),
    "CLE": ("#860038", "#FDBB30", "https://en.wikipedia.org/wiki/Special:FilePath/Cleveland_Cavaliers_logo.svg?width=512"),
    "DAL": ("#00538C", "#002B5E", "https://en.wikipedia.org/wiki/Special:FilePath/Dallas_Mavericks_logo.svg?width=512"),
    "DEN": ("#0E2240", "#FEC524", "https://en.wikipedia.org/wiki/Special:FilePath/Denver_Nuggets.svg?width=512"),
    "DET": ("#C8102E", "#1D42BA", "https://en.wikipedia.org/wiki/Special:FilePath/Logo_of_the_Detroit_Pistons.svg?width=512"),
    "GSW": ("#1D428A", "#FFC72C", "https://en.wikipedia.org/wiki/Special:FilePath/Golden_State_Warriors_logo.svg?width=512"),
    "HOU": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Houston_Rockets.svg?width=512"),
    "IND": ("#002D62", "#FDBB30", "https://en.wikipedia.org/wiki/Special:FilePath/Indiana_Pacers.svg?width=512"),
    "LAC": ("#C8102E", "#1D428A", "https://en.wikipedia.org/wiki/Special:FilePath/Los_Angeles_Clippers_(2024).svg?width=512"),
    "LAL": ("#552583", "#FDB927", "https://en.wikipedia.org/wiki/Special:FilePath/Los_Angeles_Lakers_logo.svg?width=512"),
    "MEM": ("#5D76A9", "#12173F", "https://en.wikipedia.org/wiki/Special:FilePath/Memphis_Grizzlies.svg?width=512"),
    "MIA": ("#98002E", "#F9A01B", "https://en.wikipedia.org/wiki/Special:FilePath/Miami_Heat_logo.svg?width=512"),
    "MIL": ("#00471B", "#EEE1C6", "https://en.wikipedia.org/wiki/Special:FilePath/Milwaukee_Bucks_logo.svg?width=512"),
    "MIN": ("#0C2340", "#78BE20", "https://en.wikipedia.org/wiki/Special:FilePath/Minnesota_Timberwolves_logo.svg?width=512"),
    "NOP": ("#0C2340", "#C8102E", "https://en.wikipedia.org/wiki/Special:FilePath/New_Orleans_Pelicans_logo.svg?width=512"),
    "NYK": ("#006BB6", "#F58426", "https://en.wikipedia.org/wiki/Special:FilePath/New_York_Knicks_logo.svg?width=512"),
    "OKC": ("#007AC1", "#EF3B24", "https://en.wikipedia.org/wiki/Special:FilePath/Oklahoma_City_Thunder.svg?width=512"),
    "ORL": ("#0077C0", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Orlando_Magic_logo.svg?width=512"),
    "PHI": ("#006BB6", "#ED174C", "https://en.wikipedia.org/wiki/Special:FilePath/Philadelphia_76ers_logo.svg?width=512"),
    "PHX": ("#1D1160", "#E56020", "https://en.wikipedia.org/wiki/Special:FilePath/Phoenix_Suns_logo.svg?width=512"),
    "POR": ("#E03A3E", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Portland_Trail_Blazers_logo.svg?width=512"),
    "SAC": ("#5A2D81", "#63727A", "https://en.wikipedia.org/wiki/Special:FilePath/SacramentoKings.svg?width=512"),
    "SAS": ("#C4CED4", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/San_Antonio_Spurs.svg?width=512"),
    "TOR": ("#CE1141", "#000000", "https://en.wikipedia.org/wiki/Special:FilePath/Toronto_Raptors_logo.svg?width=512"),
    "UTA": ("#002B5C", "#F9A01B", "https://en.wikipedia.org/wiki/Special:FilePath/Utah_Jazz_logo_2022.svg?width=512"),
    "WAS": ("#002B5C", "#E31837", "https://en.wikipedia.org/wiki/Special:FilePath/Washington_Wizards_logo.svg?width=512"),
}

def fetch_with_retry(endpoint_func, max_retries=3, sleep=1.5, **kwargs):
    last_error = None
    for attempt in range(max_retries):
        try:
            result = endpoint_func(timeout=60, **kwargs)
            time.sleep(0.7)
            return result.get_data_frames()
        except Exception as e:
            last_error = e
            time.sleep(sleep * (attempt + 1))
    raise last_error

def load_logo_image(url):
    if not url:
        return None
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=20) as response:
            return Image.open(BytesIO(response.read())).convert("RGBA")
    except Exception:
        return None

def pct_fmt(x, digits=0):
    if pd.isna(x):
        return "N/A"
    return f"{x:.{digits}f}%"

def num_fmt(x, digits=1):
    if pd.isna(x):
        return "N/A"
    return f"{x:.{digits}f}"

def percentile_rank(series, higher_is_better=True):
    s = pd.to_numeric(series, errors="coerce")
    return s.rank(
        pct=True,
        ascending=True if higher_is_better else False
    ) * 100

## 4. Player Lookup and Player Info

In [11]:
def find_player(player_name):
    matches = players.find_players_by_full_name(player_name)
    if not matches:
        raise ValueError(f"No NBA player found for: {player_name}")
    return matches[0]

PLAYER = find_player(PLAYER_NAME)
PLAYER_ID = PLAYER["id"]
PLAYER_DISPLAY_NAME = PLAYER["full_name"]

info_df = fetch_with_retry(commonplayerinfo.CommonPlayerInfo, player_id=PLAYER_ID)[0]
player_info = info_df.iloc[0]

POSITION = player_info.get("POSITION", "")
HEIGHT = player_info.get("HEIGHT", "")
WEIGHT = player_info.get("WEIGHT", "")
COUNTRY = player_info.get("COUNTRY", "")
SCHOOL = player_info.get("SCHOOL", "")

PLAYER_DISPLAY_NAME, PLAYER_ID, POSITION, HEIGHT, WEIGHT, COUNTRY, SCHOOL

('Andrew Wiggins', 203952, 'Forward', '6-6', '197', 'Canada', 'Kansas')

## 5. Pull League Player Stats

In [12]:
def unique_seasons(selected_season, comp_seasons):
    seasons = [selected_season] + list(comp_seasons)
    return list(dict.fromkeys(seasons))
PULL_SEASONS = unique_seasons(SEASON, COMP_SEASONS)

def get_endpoint_df(season, measure_type, per_mode):
    return fetch_with_retry(
        leaguedashplayerstats.LeagueDashPlayerStats,
        season=season,
        season_type_all_star=SEASON_TYPE,
        measure_type_detailed_defense=measure_type,
        per_mode_detailed=per_mode,
        rank="N",
        pace_adjust="N",
        plus_minus="N",
        league_id_nullable="00",
    )[0]

def keep_existing(df, cols):
    return [c for c in cols if c in df.columns]

def get_player_stats(season):
    totals = get_endpoint_df(season, "Base", "Totals")
    base = get_endpoint_df(season, "Base", PER_MODE)
    adv = get_endpoint_df(season, "Advanced", PER_MODE)
    scoring = get_endpoint_df(season, "Scoring", PER_MODE)
    totals_cols = keep_existing(totals, [
        "PLAYER_ID", "TEAM_ID", "TEAM_ABBREVIATION",
        "PLAYER_NAME", "NICKNAME", "AGE", "GP", "W", "L", "MIN"
    ])
    totals_small = totals[totals_cols].copy()
    totals_small = totals_small.rename(columns={"MIN": "TOTAL_MIN"})
    totals_small["MPG"] = totals_small["TOTAL_MIN"] / totals_small["GP"]
    base_cols = keep_existing(base, [
        "PLAYER_ID", "TEAM_ID", "TEAM_ABBREVIATION",
        "PTS", "REB", "AST", "STL", "BLK",
        "FGM", "FGA", "FG_PCT",
        "FG3M", "FG3A", "FG3_PCT",
        "FTM", "FTA", "FT_PCT",
        "OREB", "DREB", "TOV", "PF",
        "PLUS_MINUS"
    ])
    adv_cols = keep_existing(adv, [
        "PLAYER_ID", "TEAM_ID", "TEAM_ABBREVIATION",
        "OFF_RATING", "DEF_RATING", "NET_RATING",
        "AST_PCT", "AST_TO", "AST_RATIO",
        "OREB_PCT", "DREB_PCT", "REB_PCT",
        "TM_TOV_PCT", "EFG_PCT", "TS_PCT",
        "USG_PCT", "PACE", "PIE"
    ])
    scoring_cols = keep_existing(scoring, [
        "PLAYER_ID", "TEAM_ID", "TEAM_ABBREVIATION",
        "PCT_FGA_2PT", "PCT_FGA_3PT",
        "PCT_PTS_2PT", "PCT_PTS_3PT", "PCT_PTS_FB",
        "PCT_PTS_FT", "PCT_PTS_OFF_TOV",
        "PCT_AST_2PM", "PCT_UAST_2PM",
        "PCT_AST_3PM", "PCT_UAST_3PM"
    ])
    df = totals_small.copy()
    df = df.merge(
        base[base_cols],
        on=["PLAYER_ID", "TEAM_ID", "TEAM_ABBREVIATION"],
        how="left"
    )
    df = df.merge(
        adv[adv_cols],
        on=["PLAYER_ID", "TEAM_ID", "TEAM_ABBREVIATION"],
        how="left"
    )
    df = df.merge(
        scoring[scoring_cols],
        on=["PLAYER_ID", "TEAM_ID", "TEAM_ABBREVIATION"],
        how="left"
    )
    df["SEASON"] = season
    return df

all_seasons = []
for season in PULL_SEASONS:
    season_df = get_player_stats(season)
    all_seasons.append(season_df)

league_df_raw = pd.concat(all_seasons, ignore_index=True)
league_df = league_df_raw[
    (league_df_raw["GP"] >= MIN_GP) &
    (league_df_raw["MPG"] >= MIN_MINUTES_PER_GAME)
].copy()
target_row = league_df[
    (league_df["PLAYER_ID"] == PLAYER_ID) &
    (league_df["SEASON"] == SEASON)
].copy()

if target_row.empty:
    raise ValueError(
        f"{PLAYER_DISPLAY_NAME} was not found in the qualified data for {SEASON}. Check minimum games played or minuters per game.")

target = target_row.iloc[0]
TEAM_ABBR = target["TEAM_ABBREVIATION"]
PRIMARY, SECONDARY, TEAM_LOGO_URL = TEAM_COLORS.get(TEAM_ABBR, ("#222222", "#888888", None))
logo_img = load_logo_image(TEAM_LOGO_URL)

## 6. Role Context and Percentiles

In [13]:
def position_group(position):
    p = str(position).upper()
    if "GUARD" in p or p in ["G", "PG", "SG"]:
        return "Guard"
    if "CENTER" in p or p == "C":
        return "Center"
    if "FORWARD" in p or p in ["F", "SF", "PF"]:
        return "Wing/Forward"
    return "All"

def broad_profile_group(row, position=None):
    pos = str(position or "").lower()

    if "center" in pos or pos == "c":
        return "Bigs"

    if "forward-center" in pos or "center-forward" in pos:
        return "Bigs"

    reb_pct = row.get("REB_PCT", np.nan)
    blk = row.get("BLK", np.nan)
    ast_pct = row.get("AST_PCT", np.nan)
    three_rate = row.get("PCT_FGA_3PT", np.nan)

    if pd.notna(reb_pct) and reb_pct >= 0.125 and (pd.isna(ast_pct) or ast_pct < 0.22):
        return "Bigs"

    if pd.notna(blk) and blk >= 1.4 and (pd.isna(three_rate) or three_rate < 0.30):
        return "Bigs"

    return "Guards/Wings"

def inter_role(row):
    mpg = row.get("MPG", np.nan)
    usg = row.get("USG_PCT", np.nan)
    ast_pct = row.get("AST_PCT", np.nan)
    three_rate = row.get("PCT_FGA_3PT", np.nan)

    if mpg >= 30 and usg >= 0.27:
        usage = "High-usage core"
    elif mpg >= 24 and usg >= 0.22:
        usage = "Rotation creator"
    elif mpg >= 18:
        usage = "Rotation connector"
    else:
        usage = "Lower-minute role"

    if ast_pct >= 0.28:
        creation = "primary creator"
    elif ast_pct >= 0.18:
        creation = "secondary creator"
    elif three_rate >= 0.45:
        creation = "floor spacer"
    else:
        creation = "play finisher"

    return f"{usage} / {creation}"

POS_GROUP = position_group(POSITION)
TARGET_PROFILE_GROUP = broad_profile_group(target, POSITION)
IS_BIG_PROFILE = TARGET_PROFILE_GROUP == "Bigs"

league_df["PROFILE_GROUP"] = league_df.apply(broad_profile_group, axis=1)

higher_better = {
    "PTS": True,
    "TS_PCT": True,
    "USG_PCT": True,
    "FG3A": True,
    "FG3_PCT": True,
    "FG_PCT": True,
    "OREB": True,
    "OREB_PCT": True,
    "DREB": True,
    "DREB_PCT": True,
    "BLK": True,
    "AST_PCT": True,
    "AST_TO": True,
    "REB_PCT": True,
    "DEF_RATING": False,
}

PERCENTILE_METRICS = [
    "PTS", "TS_PCT", "USG_PCT", "FG3A", "FG3_PCT", "FG_PCT",
    "OREB", "OREB_PCT", "DREB", "DREB_PCT", "BLK",
    "AST_PCT", "AST_TO", "REB_PCT", "DEF_RATING",
]

for col in PERCENTILE_METRICS:
    if col in league_df.columns:
        league_df[f"{col}_PCTL"] = league_df.groupby("PROFILE_GROUP")[col].transform(
            lambda s: percentile_rank(s, higher_is_better=higher_better.get(col, True))
        )

league_df["SECOND_CHANCE_PCTL"] = league_df[
    [c for c in ["OREB_PCT_PCTL", "OREB_PCTL"] if c in league_df.columns]
].mean(axis=1)

league_df["SECOND_CHANCE"] = league_df["SECOND_CHANCE_PCTL"]

league_df["BIG_DEFENSE_PCTL"] = (
    league_df.get("BLK_PCTL", np.nan) * 0.35 +
    league_df.get("DEF_RATING_PCTL", np.nan) * 0.30 +
    league_df.get("DREB_PCT_PCTL", np.nan) * 0.20 +
    league_df.get("REB_PCT_PCTL", np.nan) * 0.15
)

league_df["BIG_DEFENSE"] = league_df["BIG_DEFENSE_PCTL"]

if IS_BIG_PROFILE:
    PROFILE_METRICS = {
        "Scoring": "PTS",
        "Efficiency": "TS_PCT",
        "Usage": "USG_PCT",
        "2nd Chance": "SECOND_CHANCE",
        "Playmaking": "AST_PCT",
        "Ball Security": "AST_TO",
        "Rebounding": "REB_PCT",
        "Defense": "BIG_DEFENSE",
    }
else:
    PROFILE_METRICS = {
        "Scoring": "PTS",
        "Efficiency": "TS_PCT",
        "Usage": "USG_PCT",
        "3PT Volume": "FG3A",
        "Playmaking": "AST_PCT",
        "Ball Security": "AST_TO",
        "Rebounding": "REB_PCT",
        "Defense": "DEF_RATING",
    }

target = league_df[
    (league_df["PLAYER_ID"] == PLAYER_ID) &
    (league_df["SEASON"] == SEASON)
].iloc[0]

TARGET_ROLE = inter_role(target)

POS_GROUP, TARGET_PROFILE_GROUP, TARGET_ROLE

('Wing/Forward', 'Guards/Wings', 'Rotation creator / play finisher')

## Export CSV for NBA Comp Dashboard

In [ ]:
# import pathlib

# dashboard_export = league_df.copy()

# dashboard_export["PROFILE_GROUP"] = np.where(
#     dashboard_export["PROFILE_GROUP"].eq("Bigs"),
#     "Big",
#     "Guard/Wing",
# )

# dashboard_export["POSITION"] = np.where(
#     dashboard_export["PROFILE_GROUP"].eq("Big"),
#     "Big",
#     "Guard/Wing",
# )

# dashboard_export["SCORING_PCTL"] = dashboard_export["PTS_PCTL"]

# dashboard_export["SHOOTING_PCTL"] = dashboard_export[
#     ["FG3A_PCTL", "FG3_PCT_PCTL", "TS_PCT_PCTL"]
# ].mean(axis=1)

# dashboard_export["PLAYMAKING_PCTL"] = dashboard_export["AST_PCT_PCTL"]

# dashboard_export["REBOUNDING_PCTL"] = dashboard_export["REB_PCT_PCTL"]

# dashboard_export["DEFENSE_PCTL"] = np.where(
#     dashboard_export["PROFILE_GROUP"].eq("Big"),
#     dashboard_export["BIG_DEFENSE_PCTL"],
#     dashboard_export["DEF_RATING_PCTL"],
# )

# dashboard_export["EFFICIENCY_PCTL"] = dashboard_export["TS_PCT_PCTL"]

# dashboard_export["RIM_PRESSURE_PCTL"] = dashboard_export[
#     ["FG_PCT_PCTL", "PTS_PCTL", "USG_PCT_PCTL"]
# ].mean(axis=1)

# dashboard_export["CREATION_PCTL"] = dashboard_export[
#     ["USG_PCT_PCTL", "AST_PCT_PCTL"]
# ].mean(axis=1)

# dashboard_export["ARCHETYPE"] = dashboard_export["PROFILE_GROUP"]

# dashboard_columns = [
#     "PLAYER_NAME",
#     "SEASON",
#     "TEAM_ABBREVIATION",
#     "POSITION",
#     "PROFILE_GROUP",
#     "AGE",
#     "PTS",
#     "REB",
#     "AST",
#     "TS_PCT",
#     "USG_PCT",
#     "SCORING_PCTL",
#     "SHOOTING_PCTL",
#     "PLAYMAKING_PCTL",
#     "REBOUNDING_PCTL",
#     "DEFENSE_PCTL",
#     "EFFICIENCY_PCTL",
#     "RIM_PRESSURE_PCTL",
#     "CREATION_PCTL",
#     "ARCHETYPE",
# ]

# player_similarity_profiles = dashboard_export[dashboard_columns].copy()

# output_path = pathlib.Path("nba_comp_dashboard/data/player_similarity_profiles.csv")
# output_path.parent.mkdir(parents=True, exist_ok=True)

# player_similarity_profiles.to_csv(output_path, index=False)

# print(f"Exported {len(player_similarity_profiles):,} player-seasons to {output_path}")
# player_similarity_profiles.head()

Exported 8,985 player-seasons to nba_comp_dashboard/data/player_similarity_profiles.csv


,PLAYER_NAME,SEASON,TEAM_ABBREVIATION,POSITION,PROFILE_GROUP,AGE,PTS,REB,AST,TS_PCT,USG_PCT,SCORING_PCTL,SHOOTING_PCTL,PLAYMAKING_PCTL,REBOUNDING_PCTL,DEFENSE_PCTL,EFFICIENCY_PCTL,RIM_PRESSURE_PCTL,CREATION_PCTL,ARCHETYPE
0,Aaron Brooks,2015-16,CHI,Guard/Wing,Guard/Wing,31.0,21.8,4.5,8.0,0.494,0.225,61.302099,48.780550,82.696988,9.796167,75.334652,16.253423,51.574384,79.091877,Guard/Wing
1,Aaron Gordon,2015-16,ORL,Big,Big,20.0,18.9,13.3,3.4,0.541,0.169,43.716300,67.351030,67.482372,41.518042,39.840315,42.430527,38.317434,54.562422,Big
4,Al Horford,2015-16,ATL,Big,Big,30.0,22.9,10.9,4.8,0.565,0.202,69.141435,78.204065,91.061800,11.012028,52.822480,59.975114,62.353104,78.494401,Big
5,Al Jefferson,2015-16,CHA,Big,Big,31.0,25.3,13.6,3.2,0.507,0.240,79.987557,20.275128,73.641642,41.518042,59.885939,18.851099,67.738145,79.147657,Big
6,Al-Farouq Aminu,2015-16,POR,Guard/Wing,Guard/Wing,25.0,17.5,10.4,2.9,0.533,0.166,30.803164,55.841193,21.858838,89.648616,45.588683,44.820505,30.658655,26.418467,Guard/Wing


## 7. Similarity Scores

In [371]:
SIM_FEATURES = [
    "PTS", "REB", "AST", "STL", "BLK", "FG3A",
    "USG_PCT", "TS_PCT", "AST_PCT", "REB_PCT",
    "TM_TOV_PCT", "PCT_FGA_3PT", "PCT_PTS_FT", "PIE",
]
SIM_FEATURES = [c for c in SIM_FEATURES if c in league_df.columns]

comp_pool = league_df.copy()

if "PROFILE_GROUP" in comp_pool.columns:
    comp_pool = comp_pool[comp_pool["PROFILE_GROUP"] == TARGET_PROFILE_GROUP].copy()

comp_pool = comp_pool[
    comp_pool["PLAYER_ID"] != PLAYER_ID
].copy()

if comp_pool.empty:
    comp_pool = league_df[
        league_df["PLAYER_ID"] != PLAYER_ID
    ].copy()

numeric_features = comp_pool[SIM_FEATURES].apply(pd.to_numeric, errors="coerce")
means = numeric_features.mean()
stds = numeric_features.std().replace(0, np.nan)

target_vec = (target[SIM_FEATURES].astype(float) - means) / stds
pool_z = (comp_pool[SIM_FEATURES].apply(pd.to_numeric, errors="coerce") - means) / stds

dist = np.sqrt(((pool_z - target_vec) ** 2).mean(axis=1))
comp_pool["DISTANCE"] = dist

max_dist = comp_pool["DISTANCE"].max()
comp_pool["SIMILARITY_SCORE"] = (
    100 if pd.isna(max_dist) or max_dist == 0
    else 100 - ((comp_pool["DISTANCE"] / max_dist) * 35)
)

comp_pool["SIMILARITY_SCORE"] = comp_pool["SIMILARITY_SCORE"].clip(0, 100)

comp_pool = (
    comp_pool
    .sort_values("SIMILARITY_SCORE", ascending=False)
    .drop_duplicates(subset=["PLAYER_ID"], keep="first")
    .copy()
)

top_comps = comp_pool.head(8).copy()
top_comp_row = top_comps.iloc[0] if len(top_comps) else None

comps_pdf = top_comps[[
    "PLAYER_NAME", "SEASON", "TEAM_ABBREVIATION", "AGE",
    "SIMILARITY_SCORE", "PTS", "REB", "AST", "TS_PCT", "USG_PCT",
]].copy()

comps_pdf["Similarity"] = comps_pdf["SIMILARITY_SCORE"].map(lambda x: f"{x:.0f}")
comps_pdf["PTS"] = comps_pdf["PTS"].map(lambda x: f"{x:.1f}")
comps_pdf["REB"] = comps_pdf["REB"].map(lambda x: f"{x:.1f}")
comps_pdf["AST"] = comps_pdf["AST"].map(lambda x: f"{x:.1f}")
comps_pdf["TS%"] = comps_pdf["TS_PCT"].map(lambda x: pct_display(x))
comps_pdf["USG%"] = comps_pdf["USG_PCT"].map(lambda x: pct_display(x))

comps_pdf = comps_pdf[[
    "PLAYER_NAME", "SEASON", "TEAM_ABBREVIATION", "AGE",
    "Similarity", "PTS", "REB", "AST", "TS%", "USG%",
]].rename(columns={
    "PLAYER_NAME": "Player",
    "TEAM_ABBREVIATION": "Team",
    "AGE": "Age",
    "SEASON": "Season",
})
comps_pdf["Age"] = comps_pdf["Age"].map(lambda x: f"{float(x):.0f}" if pd.notna(x) else "N/A")

comps_pdf

,Player,Season,Team,Age,Similarity,PTS,REB,AST,TS%,USG%
6520,Gerald Henderson,2012-13,CHA,25,95,25.5,6.0,4.3,53.1%,23.2%
13284,Glen Rice,1997-98,CHH,31,95,29.2,5.6,2.9,56.8%,24.8%
6471,DeMar DeRozan,2012-13,TOR,23,95,25.6,5.5,3.5,52.3%,23.8%
11877,Cuttino Mobley,2000-01,HOU,25,95,26.7,6.9,3.4,54.1%,24.0%
8890,Kevin Durant,2007-08,SEA,19,95,28.6,6.1,3.4,51.9%,27.3%
9011,Richard Jefferson,2007-08,NJN,28,95,29.7,5.5,4.0,57.1%,25.7%
6270,Rodney Stuckey,2013-14,DET,28,95,25.9,4.3,3.9,51.6%,24.0%
13114,Allan Houston,1997-98,NYK,27,95,27.9,5.1,3.9,53.0%,26.1%


## 8. Shot Location Profile

In [372]:
SHOT_ZONE_LABELS = [
    "Restricted Area",
    "In The Paint (Non-RA)",
    "Mid-Range",
    "Left Corner 3",
    "Right Corner 3",
    "Above the Break 3",
    "Backcourt",
]

def pct_display(x, digits=1):
    if pd.isna(x):
        return "N/A"
    x = float(x)
    if abs(x) <= 1:
        x = x * 100

    return f"{x:.{digits}f}%"

def get_mi_value(row, plain_col, zone=None, stat=None):
    if isinstance(row.index, pd.MultiIndex):
        if zone is None:
            key = ("", plain_col)
        else:
            key = (zone, stat)

        if key in row.index:
            return row[key]

    if zone is None and plain_col in row.index:
        return row[plain_col]

    return np.nan

def get_player_shot_locations(season):
    df = fetch_with_retry(
        leaguedashplayershotlocations.LeagueDashPlayerShotLocations,
        season=season,
        season_type_all_star=SEASON_TYPE,
        measure_type_simple="Base",
        per_mode_detailed="Totals",
        distance_range="By Zone",
        rank="N",
        league_id_nullable="00",
    )[0]
    rows = []
    for _, row in df.iterrows():
        player_id = get_mi_value(row, "PLAYER_ID")
        player_name = get_mi_value(row, "PLAYER_NAME")
        team_id = get_mi_value(row, "TEAM_ID")
        team_abbr = get_mi_value(row, "TEAM_ABBREVIATION")
        age = get_mi_value(row, "AGE")
        for zone in SHOT_ZONE_LABELS:
            fgm = get_mi_value(row, None, zone=zone, stat="FGM")
            fga = get_mi_value(row, None, zone=zone, stat="FGA")
            fg_pct = get_mi_value(row, None, zone=zone, stat="FG_PCT")

            rows.append({
                "PLAYER_ID": player_id,
                "PLAYER_NAME": player_name,
                "TEAM_ID": team_id,
                "TEAM_ABBREVIATION": team_abbr,
                "AGE": age,
                "ZONE": zone,
                "FGM": fgm,
                "FGA": fga,
                "FG_PCT": fg_pct,
            })
    tidy = pd.DataFrame(rows)
    if tidy.empty:
        return tidy
    tidy["FGA"] = pd.to_numeric(tidy["FGA"], errors="coerce")
    tidy["FGM"] = pd.to_numeric(tidy["FGM"], errors="coerce")
    tidy["FG_PCT"] = pd.to_numeric(tidy["FG_PCT"], errors="coerce")
    tidy["TOTAL_FGA"] = tidy.groupby("PLAYER_ID")["FGA"].transform("sum")
    tidy["FGA_FREQ"] = tidy["FGA"] / tidy["TOTAL_FGA"]

    return tidy

try:
    shot_locations = get_player_shot_locations(SEASON)
    player_shots = shot_locations[
        shot_locations["PLAYER_ID"].astype(str) == str(PLAYER_ID)
    ].copy()
    player_shots = player_shots[player_shots["FGA"] > 0].copy()
    player_shots["Freq"] = player_shots["FGA_FREQ"].map(lambda x: pct_display(x))
    player_shots["FG%"] = player_shots["FG_PCT"].map(lambda x: pct_display(x))
    shot_pdf = player_shots[["ZONE", "Freq", "FG%"]].rename(
        columns={"ZONE": "Zone"}
    ).reset_index(drop=True)
    if shot_pdf.empty:
        shot_pdf = pd.DataFrame({
            "Zone": ["No shot data"],
            "Freq": ["N/A"],
            "FG%": ["N/A"],
        })

except Exception as e:
    print(f"Shot locations unavailable: {e}")
    shot_pdf = pd.DataFrame({
        "Zone": ["Unavailable"],
        "Freq": ["N/A"],
        "FG%": ["N/A"],
    })

shot_pdf = shot_pdf[shot_pdf["Zone"] != "Backcourt"].copy()
shot_pdf

SHOT_MIN_ZONE_FGA = 10

def broad_shot_position_group(row):
    return broad_profile_group(row)

def shot_percentile(value, series):
    series = pd.to_numeric(series, errors="coerce").dropna()

    if pd.isna(value) or series.empty:
        return np.nan

    return (series <= value).mean() * 100

current_season_qualifiers = league_df[
    (league_df["SEASON"] == SEASON) &
    (league_df["GP"] >= MIN_GP) &
    (league_df["MPG"] >= MIN_MINUTES_PER_GAME)
].copy()

current_season_qualifiers["SHOT_POSITION_GROUP"] = current_season_qualifiers.apply(
    broad_shot_position_group,
    axis=1,
)

target_shot_group = current_season_qualifiers.loc[
    current_season_qualifiers["PLAYER_ID"] == PLAYER_ID,
    "SHOT_POSITION_GROUP"
].iloc[0]

shot_locations_for_pctl = shot_locations.merge(
    current_season_qualifiers[["PLAYER_ID", "SHOT_POSITION_GROUP"]],
    on="PLAYER_ID",
    how="inner",
)

shot_locations_for_pctl = shot_locations_for_pctl[
    (shot_locations_for_pctl["SHOT_POSITION_GROUP"] == target_shot_group) &
    (shot_locations_for_pctl["FGA"] >= SHOT_MIN_ZONE_FGA)
].copy()

player_shots["ZONE_FG_PCTL"] = player_shots.apply(
    lambda r: shot_percentile(
        r["FG_PCT"],
        shot_locations_for_pctl.loc[
            shot_locations_for_pctl["ZONE"] == r["ZONE"],
            "FG_PCT"
        ]
    ),
    axis=1,
)

SHOT_MIN_PLAYER_ZONE_FGA = 10

player_shots["ZONE_LOW_SAMPLE"] = (
    pd.to_numeric(player_shots["FGA"], errors="coerce") < SHOT_MIN_PLAYER_ZONE_FGA
)

shot_pdf = shot_pdf.merge(
    player_shots[["ZONE", "ZONE_FG_PCTL"]].rename(columns={"ZONE": "Zone"}),
    on="Zone",
    how="left",
)

player_shots[["ZONE", "FG_PCT", "FGA_FREQ", "ZONE_FG_PCTL"]]

,ZONE,FG_PCT,FGA_FREQ,ZONE_FG_PCTL
182,Restricted Area,0.640,0.347759,76.785714
183,In The Paint (Non-RA),0.470,0.154560,91.402715
184,Mid-Range,0.341,0.350850,19.819820
185,Left Corner 3,0.476,0.016229,88.082902
186,Right Corner 3,0.222,0.006955,6.914894
187,Above the Break 3,0.283,0.122875,13.084112
188,Backcourt,0.000,0.000773,50.000000


## 9. PDF Helper Functions

In [373]:
def wrap_cells(df, width=20):
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].astype(str).map(
            lambda x: "\n".join(textwrap.wrap(x, width=width)) if len(x) > width else x
        )
    return out

def draw_pdf_table(
    ax, df, title, title_color, header_color,
    font_size=7, header_font_size=None,
    first_col_width=0.28, wrap_width=18,
    table_height=0.86, note=None,
    note_y=0.01,
    row_scale_y=1.0,
):
    ax.axis("off")
    df = wrap_cells(df.copy(), width=wrap_width)

    if header_font_size is None:
        header_font_size = font_size + 0.45

    ax.text(
        0, 1.0, title.upper(),
        transform=ax.transAxes,
        fontsize=10.2,
        fontweight="bold",
        color=title_color,
        va="top",
        ha="left",
    )

    n_cols = len(df.columns)
    other_width = (1 - first_col_width) / max(n_cols - 1, 1)
    col_widths = [first_col_width] + [other_width] * (n_cols - 1)

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        colLoc="center",
        colWidths=col_widths,
        bbox=[0, 0.070 if note else 0, 1, table_height],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)

    if row_scale_y != 1.0:
        table.scale(1.0, row_scale_y)

    for (row, col), cell in table.get_celld().items():
        cell.set_linewidth(0.25)
        cell.set_edgecolor("#D8DEE6")

        if row == 0:
            cell.set_facecolor(header_color)
            cell.set_text_props(color="white", weight="bold", fontsize=header_font_size)
        else:
            cell.set_facecolor("#FFFFFF" if row % 2 else "#F6F8FA")
            cell.set_text_props(color="#1F2933", ha="left" if col == 0 else "center")
            if col == 0:
                cell.set_text_props(weight="bold")

    if note:
        ax.text(
            0, note_y, note,
            transform=ax.transAxes,
            fontsize=6.2,
            color="#6B7280",
            ha="left",
            va="bottom",
            style="italic",
        )

    return table

def draw_radar(ax, labels, values, color, secondary):
    labels = list(labels)
    values = np.array(values, dtype=float)
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()

    values_closed = values.tolist() + values.tolist()[:1]
    angles_closed = angles + angles[:1]

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    ax.fill(angles_closed, values_closed, color=color, alpha=0.18, zorder=2)
    ax.plot(angles_closed, values_closed, color=color, linewidth=2.4, zorder=1)
    ax.scatter(angles, values, s=24, color=secondary, edgecolor=color, linewidth=0.8, zorder=3)

    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20", "40", "60", "80", "100"], fontsize=5.8, color="#374151")
    for label in ax.get_yticklabels():
        label.set_zorder(10)
        label.set_fontweight("bold")
        label.set_bbox({
            "facecolor": "white",
            "edgecolor": "#E5E7EB",
            "linewidth": 0.25,
            "alpha": 0.88,
            "pad": 0.7,
        })

    ax.set_xticks(angles)
    ax.set_xticklabels([])

    label_pad = {
        "Efficiency": 1.12,
        "3PT Volume": 1.13,
        "2nd Chance": 1.16,
        "Ball Security": 1.12,
        "Rebounding": 1.19,
        "Defense": 1.1,
        "Usage": 1.1
    }

    default_radius = 108

    for angle, label in zip(angles, labels):
        radius = default_radius * label_pad.get(label, 1.0)

        ax.text(
            angle,
            radius,
            label,
            fontsize=6.8,
            color="#1F2933",
            fontweight="bold",
            ha="center",
            va="center",
        )
    ax.grid(color="#D8DEE6", linewidth=0.7)
    ax.spines["polar"].set_color("#CBD5E1")

SHOT_PCTL_COLORS = {
    "0-20": "#C97B7B",
    "20-40": "#E1B0A6",
    "40-60": "#D8DEE6",
    "60-80": "#AFC9B4",
    "80-100": "#6FA67A",
}

def shot_pctl_bucket(pctl):
    if pd.isna(pctl):
        return "40-60"
    if pctl < 20:
        return "0-20"
    if pctl < 40:
        return "20-40"
    if pctl < 60:
        return "40-60"
    if pctl < 80:
        return "60-80"
    return "80-100"

def shot_zone_face(row):
    if row is None:
        return "#F3F4F6"

    if bool(row.get("ZONE_LOW_SAMPLE", False)):
        return "#E5E7EB"

    bucket = shot_pctl_bucket(row.get("ZONE_FG_PCTL", np.nan))
    return SHOT_PCTL_COLORS[bucket]

def blend_with_white(hex_color, alpha=0.45):
    rgb = np.array(mcolors.to_rgb(hex_color))
    white = np.array([1, 1, 1])
    return tuple((alpha * rgb) + ((1 - alpha) * white))

def draw_half_court(ax, line_color="#64748B"):
    ax.set_xlim(-25, 25)
    ax.set_ylim(-5.5, 52)
    ax.set_aspect("equal")
    ax.axis("off")

    lw = 1.0

    ax.add_patch(MplRectangle((-25, 0), 50, 47, fill=False, edgecolor=line_color, linewidth=lw))

    ax.add_patch(MplRectangle((-8, 0), 16, 19, fill=False, edgecolor=line_color, linewidth=lw))
    ax.add_patch(MplRectangle((-6, 0), 12, 19, fill=False, edgecolor=line_color, linewidth=lw))

    ax.add_patch(Circle((0, 5.25), 0.75, fill=False, edgecolor=line_color, linewidth=lw))
    ax.plot([-3, 3], [4, 4], color=line_color, linewidth=lw)

    ax.add_patch(Arc((0, 5.25), 8, 8, theta1=0, theta2=180, edgecolor=line_color, linewidth=lw))

    ax.add_patch(Arc((0, 19), 12, 12, theta1=0, theta2=180, edgecolor=line_color, linewidth=lw))
    ax.add_patch(Arc((0, 19), 12, 12, theta1=180, theta2=360, edgecolor=line_color, linewidth=lw, linestyle="--"))

    ax.plot([-22, -22], [0, 14], color=line_color, linewidth=lw)
    ax.plot([22, 22], [0, 14], color=line_color, linewidth=lw)
    ax.add_patch(Arc((0, 5.25), 47.5, 47.5, theta1=22, theta2=158, edgecolor=line_color, linewidth=lw))

    ax.add_patch(Arc((0, 47), 12, 12, theta1=180, theta2=360, edgecolor=line_color, linewidth=lw))


def zone_row(shot_zone_df, zone):
    match = shot_zone_df.loc[shot_zone_df["Zone"] == zone]
    if match.empty:
        return None
    return match.iloc[0]


def zone_label(row):
    if row is None:
        return "N/A"

    fg = float(row["FG_PCT"]) * 100
    freq = float(row["FGA_FREQ"]) * 100

    if bool(row.get("ZONE_LOW_SAMPLE", False)):
        return f"{fg:.1f}%\nLow Vol"

    return f"{fg:.1f}%\n({freq:.1f}%)"


def zone_face(primary, row, max_freq):
    if row is None:
        return "#F3F4F6"

    freq = float(row["FGA_FREQ"])
    strength = 0.24 + 0.34 * (freq / max_freq)
    return blend_with_white(primary, alpha=strength)


def add_patch(ax, patch, face):
    patch.set_facecolor(face)
    patch.set_edgecolor(face)
    patch.set_linewidth(0.2)
    patch.set_alpha(1.0)
    ax.add_patch(patch)
    return patch


def restricted_area_patch():
    radius = 3.8
    center_y = 5.25

    verts = [(-8, 0), (8, 0), (8, center_y)]

    for angle in np.linspace(0, 180, 50):
        rad = np.deg2rad(angle)
        verts.append((radius * np.cos(rad), center_y + radius * np.sin(rad)))

    verts.append((-8, center_y))
    verts.append((-8, 0))

    codes = [Path.MOVETO] + [Path.LINETO] * (len(verts) - 2) + [Path.CLOSEPOLY]
    return PathPatch(Path(verts, codes))


def above_break_three_patch():
    r = 23.75
    cx, cy = 0, 5.25

    # Inner boundary follows the actual 3PT arc. Outer boundary is the court:
    # side lines, half court line, and the corner-break verticals.
    arc_points = []
    for angle in np.linspace(158, 22, 70):
        rad = np.deg2rad(angle)
        arc_points.append((cx + r * np.cos(rad), cy + r * np.sin(rad)))

    verts = [
        (-25, 14),
        (-25, 47),
        (25, 47),
        (25, 14),
        (22, 14),
    ]

    verts.extend(arc_points)
    verts.extend([
        (-22, 14),
        (-25, 14),
    ])

    return Polygon(verts, closed=True)


def midrange_patches():
    # Draw midrange slightly underneath the paint/RA so tiny seams get covered.
    return [
        MplRectangle((-22.15, 0), 14.35, 14.35),
        MplRectangle((7.80, 0), 14.35, 14.35),
        Wedge((0, 5.25), 24.10, 19.5, 160.5, width=11.10),
    ]

def zone_text_box():
    return {
        "boxstyle": "round,pad=0.18,rounding_size=0.04",
        "facecolor": "white",
        "edgecolor": "#CBD5E1",
        "linewidth": 0.45,
        "alpha": 0.78,
    }

def draw_shot_court(ax, shot_zone_df, primary, secondary):
    ax.set_xlim(-25, 25)
    ax.set_ylim(-5.5, 52)
    ax.set_aspect("equal")
    ax.axis("off")

    ax.text(
        0.0, 1.035, "SHOT LOCATION MAP",
        transform=ax.transAxes,
        fontsize=10.0,
        fontweight="bold",
        color=primary,
        va="top",
        ha="left",
        clip_on=False,
    )

    ax.text(
        0.0, 0.970, "Labels: FG% (FGA freq)",
        transform=ax.transAxes,
        fontsize=6.3,
        color="#6B7280",
        ha="left",
        va="top",
        clip_on=False,
    )

    if shot_zone_df.empty:
        draw_half_court(ax)
        ax.text(0, 24, "Shot data unavailable", ha="center", va="center", fontsize=8, color="#6B7280")
        return

    max_freq = max(float(shot_zone_df["FGA_FREQ"].max()), 0.01)

    ra = zone_row(shot_zone_df, "Restricted Area")
    paint = zone_row(shot_zone_df, "In The Paint (Non-RA)")
    mid = zone_row(shot_zone_df, "Mid-Range")
    left_corner = zone_row(shot_zone_df, "Left Corner 3")
    right_corner = zone_row(shot_zone_df, "Right Corner 3")
    above_break = zone_row(shot_zone_df, "Above the Break 3")

    ra_color = shot_zone_face(ra)
    paint_color = shot_zone_face(paint)
    mid_color = shot_zone_face(mid)
    left_corner_color = shot_zone_face(left_corner)
    right_corner_color = shot_zone_face(right_corner)
    above_break_color = shot_zone_face(above_break)

    # 3PT zones. Above-break is bounded by the court, so it cannot shade above half court.
    add_patch(ax, above_break_three_patch(), above_break_color)
    add_patch(ax, MplRectangle((-25, 0), 3.15, 14.25), left_corner_color)
    add_patch(ax, MplRectangle((21.85, 0), 3.15, 14.25), right_corner_color)

    # Midrange. Slightly oversized pieces remove hairline white gaps.
    for patch in midrange_patches():
        add_patch(ax, patch, mid_color)

    # Paint first as a full paint block. Restricted area then sits on top.
    # This fills the tiny white gaps around the rim section.
    add_patch(ax, MplRectangle((-8, 0), 16, 19), paint_color)
    add_patch(ax, restricted_area_patch(), ra_color)

    # Court lines on top.
    draw_half_court(ax, line_color="#64748B")

    ax.text(
        0, -3.55, zone_label(ra),
        ha="center", va="center",
        fontsize=6.8,
        color="#111827",
        fontweight="bold",
        clip_on=False,
    )

    ax.text(
        0, 14.4, zone_label(paint),
        ha="center", va="center",
        fontsize=6.8,
        color="#111827",
        fontweight="bold",
        bbox=zone_text_box(),
    )

    ax.text(
        0, 25.2, zone_label(mid),
        ha="center", va="center",
        fontsize=6.8,
        color="#111827",
        fontweight="bold",
        bbox=zone_text_box(),
    )

    ax.text(
        0, 38.0, zone_label(above_break),
        ha="center", va="center",
        fontsize=6.8,
        color="#111827",
        fontweight="bold",
        bbox=zone_text_box(),
    )

    ax.text(
        -27.0, 7.0, zone_label(left_corner),
        ha="right", va="center",
        fontsize=6.5,
        color="#111827",
        fontweight="bold",
        clip_on=False,
    )

    ax.text(
        27.0, 7.0, zone_label(right_corner),
        ha="left", va="center",
        fontsize=6.5,
        color="#111827",
        fontweight="bold",
        clip_on=False,
    )

def draw_shot_percentile_legend(ax, shot_group_label):
    ax.axis("off")

    ax.text(
        0.0, 0.98, "ZONE EFFICIENCY",
        transform=ax.transAxes,
        fontsize=7.2,
        fontweight="bold",
        color="#1F2933",
        ha="left",
        va="top",
    )

    ax.text(
        0.0, 0.895,
        f"FG% percentile\nvs {shot_group_label}",
        transform=ax.transAxes,
        fontsize=5.8,
        color="#6B7280",
        ha="left",
        va="top",
    )

    legend_rows = [
        ("80-100", "Elite"),
        ("60-80", "Above Avg"),
        ("40-60", "Average"),
        ("20-40", "Below Avg"),
        ("0-20", "Poor"),
    ]

    y = 0.710

    for bucket, label in legend_rows:
        ax.add_patch(
            MplRectangle(
                (0.00, y - 0.025),
                0.105,
                0.050,
                transform=ax.transAxes,
                facecolor=SHOT_PCTL_COLORS[bucket],
                edgecolor="#CBD5E1",
                linewidth=0.4,
            )
        )

        ax.text(
            0.135, y,
            f"{bucket}%",
            transform=ax.transAxes,
            fontsize=5.6,
            color="#1F2933",
            ha="left",
            va="center",
            fontweight="bold",
        )

        ax.text(
            0.735, y,
            label,
            transform=ax.transAxes,
            fontsize=5.6,
            color="#4B5563",
            ha="left",
            va="center",
        )

        y -= 0.115

def draw_top_comp_metric_bars(ax, metric_df, primary, secondary):
    ax.axis("off")
    ax.text(0, 1.03, "TOP COMP STAT COMPARISON", transform=ax.transAxes,
            fontsize=10.2, fontweight="bold", color=primary, va="bottom", ha="left")

    if metric_df.empty:
        ax.text(0.5, 0.45, "Comp data unavailable", ha="center", va="center", fontsize=8, color="#6B7280")
        return

    labels = metric_df["Metric"].tolist()
    player_vals = metric_df["Player"].values
    comp_vals = metric_df["Comp"].values

    y = np.arange(len(labels))[::-1]
    max_val = max(np.nanmax(player_vals), np.nanmax(comp_vals), 1) * 1.18

    ax.set_xlim(0, max_val)
    ax.set_ylim(-0.7, len(labels) - 0.2)

    for i, label in enumerate(labels):
        yy = y[i]
        ax.barh(yy + 0.13, player_vals[i], height=0.22, color=primary, alpha=0.85)
        ax.barh(yy - 0.13, comp_vals[i], height=0.22, color="#9CA3AF", alpha=0.85)

        ax.text(0, yy + 0.42, label, fontsize=6.8, color="#1F2933", ha="left", va="bottom", fontweight="bold")
        ax.text(player_vals[i] + max_val * 0.015, yy + 0.13, metric_df["Player Label"].iloc[i],
                fontsize=6.4, color="#1F2933", va="center")
        ax.text(comp_vals[i] + max_val * 0.015, yy - 0.13, metric_df["Comp Label"].iloc[i],
                fontsize=6.4, color="#4B5563", va="center")

    ax.text(0.00, -0.10, PLAYER_DISPLAY_NAME, transform=ax.transAxes, fontsize=6.3, color=primary, fontweight="bold")
    ax.text(0.42, -0.10, metric_df.attrs.get("comp_name", "Top comp"), transform=ax.transAxes,
            fontsize=6.3, color="#4B5563", fontweight="bold")
    
def pctl_value(row, col):
    val = row.get(f"{col}_PCTL", np.nan)
    return float(val) if pd.notna(val) else np.nan

def role_meter_score(values):
    clean = [v for v in values if pd.notna(v)]
    if not clean:
        return 50
    return int(np.clip(np.mean(clean), 0, 100))

def meter_to_10(score):
    return int(np.clip(round(score / 10), 1, 10))

def badge_level(score):
    if score >= 80:
        return "Gold"
    if score >= 65:
        return "Silver"
    if score >= 50:
        return "Bronze"
    return "Growth"

def badge_color(level):
    return {
        "Gold": "#D8A733",
        "Silver": "#A7B0BA",
        "Bronze": "#B97745",
        "Growth": "#E5E7EB",
    }.get(level, "#E5E7EB")

def readable_metric_label(area):
    return {
        "Scoring": "Scoring volume",
        "Efficiency": "Scoring efficiency",
        "Usage": "On-ball usage",
        "3PT Volume": "3PT volume",
        "2nd Chance": "Second-chance pressure",
        "Playmaking": "Playmaking",
        "Ball Security": "Ball security",
        "Rebounding": "Rebounding",
        "Defense": "Defensive results",
    }.get(area, area)

def is_big_profile(target, position):
    pos = str(position).lower()

    reb_pct = target.get("REB_PCT", np.nan)
    blk = target.get("BLK", np.nan)
    ast_pct = target.get("AST_PCT", np.nan)
    three_rate = target.get("PCT_FGA_3PT", np.nan)

    if "center" in pos or pos == "c":
        return True

    if "forward-center" in pos or "center-forward" in pos:
        return True

    if pd.notna(reb_pct) and reb_pct >= 0.125 and (pd.isna(ast_pct) or ast_pct < 0.22):
        return True

    if pd.notna(blk) and blk >= 1.7 and (pd.isna(three_rate) or three_rate < 0.30):
        return True

    return False

def build_big_archetype_data(target, top_comps, profile_detail_df):
    finisher = role_meter_score([
        pctl_value(target, "TS_PCT"),
        pctl_value(target, "PTS"),
        pctl_value(target, "FG_PCT"),
    ])

    rebounder = role_meter_score([
        pctl_value(target, "REB_PCT"),
        pctl_value(target, "DREB_PCT"),
    ])

    second_chance = role_meter_score([
        pctl_value(target, "SECOND_CHANCE"),
        pctl_value(target, "OREB_PCT"),
        pctl_value(target, "OREB"),
    ])

    rim_defense = role_meter_score([
        pctl_value(target, "BIG_DEFENSE"),
        pctl_value(target, "BLK"),
        pctl_value(target, "DEF_RATING"),
    ])

    playmaking = role_meter_score([
        pctl_value(target, "AST_PCT"),
        pctl_value(target, "AST_TO"),
    ])

    role_fit = role_meter_score([
        finisher,
        rebounder,
        second_chance,
        rim_defense,
        playmaking,
    ])

    archetype_name = "ROLL + REBOUND BIG"
    archetype_subtitle = "Interior Finisher / Glass Cleaner"

    if rim_defense >= 75 and rebounder >= 70 and finisher >= 60:
        archetype_name = "TWO-WAY INTERIOR BIG"
        archetype_subtitle = "Rim Protector / Play Finisher"
    elif finisher >= 75 and rebounder >= 75:
        archetype_name = "ROLL + REBOUND BIG"
        archetype_subtitle = "Vertical Finisher / Glass Cleaner"
    elif rim_defense >= 75 and second_chance >= 65:
        archetype_name = "DEFENSIVE ANCHOR"
        archetype_subtitle = "Rim Protector / Glass Cleaner"
    elif playmaking >= 65 and rebounder >= 60:
        archetype_name = "CONNECTIVE BIG"
        archetype_subtitle = "Short-Roll Passer / Rebounder"
    elif second_chance >= 80 and rebounder >= 70:
        archetype_name = "SECOND-CHANCE BIG"
        archetype_subtitle = "Glass Pressure / Play Finisher"
    elif rebounder >= 80:
        archetype_name = "GLASS CLEANER"
        archetype_subtitle = "Rebounding Big / Interior Finisher"

    meters = [
        ("Finisher", finisher),
        ("Rebounder", rebounder),
        ("2nd Chance", second_chance),
        ("Interior Def", rim_defense),
        ("Playmaking", playmaking),
    ]

    badge_candidates = [
        ("Paint Finisher", pctl_value(target, "TS_PCT")),
        ("Glass Cleaner", pctl_value(target, "REB_PCT")),
        ("Second Chance", pctl_value(target, "SECOND_CHANCE")),
        ("Roll Pressure", pctl_value(target, "PTS")),
        ("Rim Deterrence", pctl_value(target, "BLK")),
        ("Interior Anchor", pctl_value(target, "BIG_DEFENSE")),
        ("Short-Roll Feel", pctl_value(target, "AST_PCT")),
    ]

    badges = []
    for name, score in badge_candidates:
        if pd.notna(score):
            badges.append((name, badge_level(score), score))

    badges = sorted(badges, key=lambda x: x[2], reverse=True)[:4]

    comp_group = top_comps.head(5).copy()
    comp_strengths = []
    comp_gaps = []

    if len(comp_group):
        big_comp_metrics = [
            ("Interior finishing", "TS_PCT_PCTL"),
            ("Rebounding", "REB_PCT_PCTL"),
            ("Second-chance pressure", "SECOND_CHANCE_PCTL"),
            ("Defensive impact", "BIG_DEFENSE_PCTL"),
            ("Playmaking", "AST_PCT_PCTL"),
        ]

        comp_summary = []

        for label, pctl_col in big_comp_metrics:
            if pctl_col in comp_group.columns:
                avg_pctl = comp_group[pctl_col].mean()
                if pd.notna(avg_pctl):
                    comp_summary.append((label, avg_pctl))

        comp_summary = sorted(comp_summary, key=lambda x: x[1], reverse=True)

        comp_strengths = [x[0] for x in comp_summary[:3]]
        comp_gaps = [x[0] for x in comp_summary[-2:]]

    if not comp_strengths:
        comp_strengths = ["Rim finishing", "Rebounding", "Second-chance pressure"]

    if not comp_gaps:
        comp_gaps = ["Playmaking", "Defensive impact"]

    swing_options = [
        ("Finishing efficiency", finisher),
        ("Defensive rebounding consistency", rebounder),
        ("Second-chance creation", second_chance),
        ("Defensive impact consistency", rim_defense),
        ("Short-roll decision making", playmaking),
    ]

    swing_options = [x for x in swing_options if pd.notna(x[1])]

    if swing_options:
        swing_skill = sorted(swing_options, key=lambda x: x[1])[0][0]
    else:
        swing_skill = "Decision-making consistency"

    return {
        "archetype_name": archetype_name,
        "archetype_subtitle": archetype_subtitle,
        "role_fit": role_fit,
        "meters": meters,
        "badges": badges,
        "comp_strengths": comp_strengths,
        "comp_gaps": comp_gaps,
        "swing_skill": swing_skill,
    }

def is_wing_forward_profile(target, position):
    pos = str(position).lower()

    if is_big_profile(target, position):
        return False

    wing_terms = ["forward", "guard-forward", "forward-guard", "sf", "pf", "f"]

    return any(term in pos for term in wing_terms)


def build_guard_archetype_data(target, top_comps, profile_detail_df):
    creator = role_meter_score([
        pctl_value(target, "USG_PCT"),
        pctl_value(target, "AST_PCT"),
        pctl_value(target, "PTS"),
    ])

    connector = role_meter_score([
        pctl_value(target, "AST_TO"),
        pctl_value(target, "AST_PCT"),
        100 - abs(pctl_value(target, "USG_PCT") - 65)
        if pd.notna(pctl_value(target, "USG_PCT")) else np.nan,
    ])

    shooter = role_meter_score([
        pctl_value(target, "FG3A"),
        pctl_value(target, "FG3_PCT"),
        pctl_value(target, "TS_PCT"),
    ])

    defender = role_meter_score([
        pctl_value(target, "DEF_RATING"),
    ])

    rebounder = role_meter_score([
        pctl_value(target, "REB_PCT"),
    ])

    role_fit = role_meter_score([creator, connector, shooter, defender, rebounder])

    archetype_name = "COMBO GUARD"
    archetype_subtitle = "Secondary Creator / Pressure Guard"

    if creator >= 80 and shooter >= 65:
        archetype_name = "SCORING CREATOR"
        archetype_subtitle = "On-Ball Guard / Shot Maker"
    elif creator >= 75 and connector >= 70:
        archetype_name = "LEAD CREATOR"
        archetype_subtitle = "Primary Handler / Table Setter"
    elif shooter >= 78 and creator < 70:
        archetype_name = "SPACING GUARD"
        archetype_subtitle = "Off-Ball Shooter / Secondary Handler"
    elif defender >= 70 and connector >= 60:
        archetype_name = "TWO-WAY GUARD"
        archetype_subtitle = "Point-of-Attack Defender / Connector"
    elif connector >= 72:
        archetype_name = "CONNECTOR GUARD"
        archetype_subtitle = "Secondary Creator / Decision Maker"
    elif creator >= 70:
        archetype_name = "PRESSURE GUARD"
        archetype_subtitle = "Downhill Creator / Scoring Guard"

    meters = [
        ("Creator", creator),
        ("Connector", connector),
        ("Shooter", shooter),
        ("Defender", defender),
        ("Rebounder", rebounder),
    ]

    badge_candidates = [
        ("Rim Pressure", pctl_value(target, "PTS")),
        ("Secondary Play", pctl_value(target, "AST_PCT")),
        ("Ball Control", pctl_value(target, "AST_TO")),
        ("Shooting Growth", pctl_value(target, "TS_PCT")),
        ("Defensive Tools", pctl_value(target, "DEF_RATING")),
        ("Guard Size", 65 if "guard" in str(POSITION).lower() and HEIGHT not in ["", "N/A"] else np.nan),
    ]

    badges = []
    for name, score in badge_candidates:
        if pd.notna(score):
            badges.append((name, badge_level(score), score))

    badges = sorted(badges, key=lambda x: x[2], reverse=True)[:4]

    comp_group = top_comps.head(5).copy()
    comp_strengths = []
    comp_gaps = []

    if len(comp_group):
        comp_summary = []

        for _, row in profile_detail_df.iterrows():
            area = row["Area"]
            col = row["Column"]
            pctl_col = f"{col}_PCTL"

            if pctl_col not in comp_group.columns:
                continue

            avg_pctl = comp_group[pctl_col].mean()

            if pd.notna(avg_pctl):
                comp_summary.append((area, avg_pctl))

        comp_summary = sorted(comp_summary, key=lambda x: x[1], reverse=True)

        comp_strengths = [readable_metric_label(x[0]) for x in comp_summary[:3]]
        comp_gaps = [readable_metric_label(x[0]) for x in comp_summary[-2:]]

    if not comp_strengths:
        comp_strengths = ["Creation profile", "Scoring volume", "On-ball reps"]

    if not comp_gaps:
        comp_gaps = ["Shooting efficiency", "Consistency"]

    swing_options = [
        ("Shooting consistency", shooter),
        ("Decision-making consistency", connector),
        ("Defensive impact stability", defender),
        ("Scoring efficiency", pctl_value(target, "TS_PCT")),
        ("Rebounding physicality", rebounder),
    ]

    swing_options = [x for x in swing_options if pd.notna(x[1])]

    if swing_options:
        swing_skill = sorted(swing_options, key=lambda x: x[1])[0][0]
    else:
        swing_skill = "Decision-making consistency"

    return {
        "archetype_name": archetype_name,
        "archetype_subtitle": archetype_subtitle,
        "role_fit": role_fit,
        "meters": meters,
        "badges": badges,
        "comp_strengths": comp_strengths,
        "comp_gaps": comp_gaps,
        "swing_skill": swing_skill,
    }


def build_wing_forward_archetype_data(target, top_comps, profile_detail_df):
    scoring = role_meter_score([
        pctl_value(target, "PTS"),
        pctl_value(target, "USG_PCT"),
        pctl_value(target, "TS_PCT"),
    ])

    slashing = role_meter_score([
        pctl_value(target, "PTS"),
        pctl_value(target, "FG_PCT"),
        pctl_value(target, "TS_PCT"),
    ])

    shooting = role_meter_score([
        pctl_value(target, "FG3A"),
        pctl_value(target, "FG3_PCT"),
        pctl_value(target, "TS_PCT"),
    ])

    defense = role_meter_score([
        pctl_value(target, "DEF_RATING"),
        pctl_value(target, "REB_PCT"),
    ])

    connector = role_meter_score([
        pctl_value(target, "AST_PCT"),
        pctl_value(target, "AST_TO"),
        100 - abs(pctl_value(target, "USG_PCT") - 60)
        if pd.notna(pctl_value(target, "USG_PCT")) else np.nan,
    ])

    rebounder = role_meter_score([
        pctl_value(target, "REB_PCT"),
        pctl_value(target, "DREB_PCT"),
    ])

    role_fit = role_meter_score([
        scoring,
        slashing,
        shooting,
        defense,
        connector,
        rebounder,
    ])

    archetype_name = "SCORING WING"
    archetype_subtitle = "Shot Creator / Downhill Scorer"

    if scoring >= 82 and slashing >= 65:
        archetype_name = "PRIMARY SCORING WING"
        archetype_subtitle = "High-Usage Wing / Downhill Scorer"
    elif defense >= 72 and shooting >= 65:
        archetype_name = "TWO-WAY WING"
        archetype_subtitle = "Defensive Wing / Floor Spacer"
    elif shooting >= 76 and defense >= 55:
        archetype_name = "3-AND-D WING"
        archetype_subtitle = "Spot-Up Shooter / Defensive Forward"
    elif connector >= 70 and defense >= 55:
        archetype_name = "CONNECTOR FORWARD"
        archetype_subtitle = "Decision Maker / Versatile Forward"
    elif slashing >= 75 and scoring >= 65:
        archetype_name = "SLASHING WING"
        archetype_subtitle = "Rim Pressure / Transition Scorer"
    elif rebounder >= 70 and slashing >= 60:
        archetype_name = "BIG WING FINISHER"
        archetype_subtitle = "Physical Forward / Play Finisher"
    elif shooting >= 72:
        archetype_name = "SPACING WING"
        archetype_subtitle = "Off-Ball Shooter / Floor Spacer"
    elif defense >= 72:
        archetype_name = "DEFENSIVE WING"
        archetype_subtitle = "Versatile Defender / Transition Finisher"

    meters = [
        ("Scoring", scoring),
        ("Slashing", slashing),
        ("Shooting", shooting),
        ("Defense", defense),
        ("Connector", connector),
    ]

    badge_candidates = [
        ("Rim Pressure", slashing),
        ("Scoring Pop", scoring),
        ("Shooting Growth", shooting),
        ("Defensive Tools", defense),
        ("Connector Feel", connector),
        ("Wing Rebounder", rebounder),
        ("Transition Threat", role_meter_score([pctl_value(target, "PTS"), pctl_value(target, "FG_PCT")])),
    ]

    badges = []
    for name, score in badge_candidates:
        if pd.notna(score):
            badges.append((name, badge_level(score), score))

    badges = sorted(badges, key=lambda x: x[2], reverse=True)[:4]

    comp_group = top_comps.head(5).copy()
    comp_strengths = []
    comp_gaps = []

    if len(comp_group):
        wing_comp_metrics = [
            ("Scoring volume", "PTS_PCTL"),
            ("Scoring efficiency", "TS_PCT_PCTL"),
            ("On-ball usage", "USG_PCT_PCTL"),
            ("Shooting volume", "FG3A_PCTL"),
            ("Playmaking", "AST_PCT_PCTL"),
            ("Rebounding", "REB_PCT_PCTL"),
            ("Defensive impact", "DEF_RATING_PCTL"),
        ]

        comp_summary = []

        for label, pctl_col in wing_comp_metrics:
            if pctl_col in comp_group.columns:
                avg_pctl = comp_group[pctl_col].mean()
                if pd.notna(avg_pctl):
                    comp_summary.append((label, avg_pctl))

        comp_summary = sorted(comp_summary, key=lambda x: x[1], reverse=True)

        comp_strengths = [x[0] for x in comp_summary[:3]]
        comp_gaps = [x[0] for x in comp_summary[-2:]]

    if not comp_strengths:
        comp_strengths = ["Scoring volume", "Rim pressure", "Wing athleticism"]

    if not comp_gaps:
        comp_gaps = ["Shooting consistency", "Playmaking"]

    swing_options = [
        ("Shooting consistency", shooting),
        ("Playmaking growth", connector),
        ("Defensive impact stability", defense),
        ("Scoring efficiency", pctl_value(target, "TS_PCT")),
        ("Rebounding physicality", rebounder),
    ]

    swing_options = [x for x in swing_options if pd.notna(x[1])]

    if swing_options:
        swing_skill = sorted(swing_options, key=lambda x: x[1])[0][0]
    else:
        swing_skill = "Shooting consistency"

    return {
        "archetype_name": archetype_name,
        "archetype_subtitle": archetype_subtitle,
        "role_fit": role_fit,
        "meters": meters,
        "badges": badges,
        "comp_strengths": comp_strengths,
        "comp_gaps": comp_gaps,
        "swing_skill": swing_skill,
    }


def build_role_archetype_data(target, top_comps, profile_detail_df):
    if is_big_profile(target, POSITION):
        return build_big_archetype_data(target, top_comps, profile_detail_df)

    if is_wing_forward_profile(target, POSITION):
        return build_wing_forward_archetype_data(target, top_comps, profile_detail_df)

    return build_guard_archetype_data(target, top_comps, profile_detail_df)

def draw_segment_meter(ax, x, y, label, score, primary):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=6.5,
        color="#1F2933",
        ha="left",
        va="center",
        fontweight="bold",
    )

    n = 10
    filled = meter_to_10(score)

    # Meters pulled closer to labels and slightly shorter.
    start_x = x + 0.235
    w = 0.018
    gap = 0.004

    for i in range(n):
        fill = primary if i < filled else "#E5E7EB"
        ax.add_patch(
            MplRectangle(
                (start_x + i * (w + gap), y - 0.013),
                w,
                0.026,
                transform=ax.transAxes,
                facecolor=fill,
                edgecolor="#CBD5E1",
                linewidth=0.25,
            )
        )

    ax.text(
        start_x + n * (w + gap) + 0.008,
        y,
        f"{filled}/10",
        transform=ax.transAxes,
        fontsize=5.9,
        color="#374151",
        ha="left",
        va="center",
        fontweight="bold",
    )

def draw_badge(ax, x, y, name, level):
    color = badge_color(level)

    ax.add_patch(
        FancyBboxPatch(
            (x, y - 0.022),
            0.285,
            0.043,
            boxstyle="round,pad=0.004,rounding_size=0.010",
            transform=ax.transAxes,
            facecolor=blend_with_white(color, 0.30) if level != "Growth" else "#F3F4F6",
            edgecolor=color if level != "Growth" else "#CBD5E1",
            linewidth=0.65,
        )
    )

    ax.text(
        x + 0.010, y, name.upper(),
        transform=ax.transAxes,
        fontsize=5.25,
        color="#1F2933",
        ha="left",
        va="center",
        fontweight="bold",
    )

def draw_role_archetype_card(ax, data, primary, secondary):
    ax.axis("off")

    ax.add_patch(
        MplRectangle(
            (0, 0), 1, 1,
            transform=ax.transAxes,
            facecolor="#FFFFFF",
            edgecolor="#D8DEE6",
            linewidth=0.9,
        )
    )

    ax.text(
        0.040, 0.935, "ROLE ARCHETYPE",
        transform=ax.transAxes,
        fontsize=10.2,
        color=primary,
        ha="left",
        va="top",
        fontweight="bold",
    )

    ax.text(
        0.040, 0.845, data["archetype_name"],
        transform=ax.transAxes,
        fontsize=11.8,
        color="#1F2933",
        ha="left",
        va="top",
        fontweight="black",
    )

    ax.text(
        0.040, 0.758, data["archetype_subtitle"],
        transform=ax.transAxes,
        fontsize=6.7,
        color="#4B5563",
        ha="left",
        va="top",
        fontweight="bold",
    )

    # Bigger fit score box; only FIT + number inside.
    ax.add_patch(
        FancyBboxPatch(
            (0.760, 0.750), 0.175, 0.155,
            boxstyle="round,pad=0.006,rounding_size=0.014",
            transform=ax.transAxes,
            facecolor="#F7FBFF",
            edgecolor=primary,
            linewidth=0.9,
        )
    )

    ax.text(
        0.848, 0.870, "FIT",
        transform=ax.transAxes,
        fontsize=6.1,
        color="#4B5563",
        ha="center",
        va="center",
        fontweight="bold",
    )

    ax.text(
        0.848, 0.790, f"{data['role_fit']}",
        transform=ax.transAxes,
        fontsize=17.0,
        color=primary,
        ha="center",
        va="center",
        fontweight="black",
    )

    # Outside the box.
    ax.text(
        0.848, 0.715, "Archetype Match",
        transform=ax.transAxes,
        fontsize=5.4,
        color="#6B7280",
        ha="center",
        va="center",
        fontweight="bold",
    )

    ax.text(
        0.040, 0.635, "ROLE METERS",
        transform=ax.transAxes,
        fontsize=7.0,
        color=primary,
        ha="left",
        va="center",
        fontweight="bold",
    )

    meter_y = [0.565, 0.505, 0.445, 0.385, 0.325]
    for (label, score), y in zip(data["meters"], meter_y):
        draw_segment_meter(ax, 0.040, y, label, score, primary)

    ax.text(
        0.650, 0.635, "BADGES",
        transform=ax.transAxes,
        fontsize=7.0,
        color=primary,
        ha="left",
        va="center",
        fontweight="bold",
    )

    # More vertical space between the four max badges.
    badge_positions = [
        (0.650, 0.565),
        (0.650, 0.485),
        (0.650, 0.405),
        (0.650, 0.325),
    ]

    for (name, level, score), (x, y) in zip(data["badges"], badge_positions):
        draw_badge(ax, x, y, name, level)

    ax.add_patch(
        MplRectangle(
            (0.020, 0.050), 0.955, 0.160,
            transform=ax.transAxes,
            facecolor="#F6F8FA",
            edgecolor="#E5E7EB",
            linewidth=0.6,
        )
    )

    ax.text(
        0.03, 0.180, "COMP GROUP EDGE",
        transform=ax.transAxes,
        fontsize=6.2,
        color=primary,
        ha="left",
        va="center",
        fontweight="bold",
    )

    ax.text(
        0.03, 0.132, " / ".join(data["comp_strengths"][:3]),
        transform=ax.transAxes,
        fontsize=6.1,
        color="#1F2933",
        ha="left",
        va="center",
        fontweight="bold",
    )

    ax.text(
        0.03, 0.082, f"DEVELOPMENT SWING: {data['swing_skill']}",
        transform=ax.transAxes,
        fontsize=6.2,
        color="#374151",
        ha="left",
        va="center",
        fontweight="bold",
    )

## 10. Build PDF Tables

In [374]:
TEAM_FULL_NAME = TEAM_ABBR_LOOKUP.get(TEAM_ABBR, {}).get("full_name", TEAM_ABBR)

profile_rows = []

for label, col in PROFILE_METRICS.items():
    if col not in league_df.columns:
        continue

    pctl = target.get(f"{col}_PCTL", np.nan)

    if pd.notna(pctl):
        profile_rows.append({
            "Area": label,
            "Column": col,
            "Player Pctl": float(pctl),
        })

profile_detail_df = pd.DataFrame(profile_rows)

role_archetype_data = build_role_archetype_data(
    target,
    top_comps,
    profile_detail_df,
)

snapshot_pdf = pd.DataFrame({
    "Metric": [
        "Team", "Position", "Role Context", "Age",
        "Games", "Minutes/G", "Per 100 Line", "TS%", "USG%",
    ],
    "Value": [
        TEAM_FULL_NAME,
        POSITION,
        role_archetype_data["archetype_subtitle"],
        f"{target['AGE']:.0f}",
        f"{target['GP']:.0f}",
        f"{target['MPG']:.1f}",
        f"{target['PTS']:.1f} PTS / {target['REB']:.1f} REB / {target['AST']:.1f} AST",
        pct_display(target["TS_PCT"]) if "TS_PCT" in target.index else "N/A",
        pct_display(target["USG_PCT"]) if "USG_PCT" in target.index else "N/A",
    ],
})

radar_labels = profile_detail_df["Area"].tolist()
radar_values = profile_detail_df["Player Pctl"].values

try:
    shot_chart_df = player_shots.copy()
    shot_chart_df = shot_chart_df[shot_chart_df["ZONE"] != "Backcourt"].copy()
    shot_chart_df = shot_chart_df.rename(columns={"ZONE": "Zone"})

    keep_shot_cols = ["Zone", "FGA_FREQ", "FG_PCT", "ZONE_FG_PCTL", "ZONE_LOW_SAMPLE"]
    keep_shot_cols = [c for c in keep_shot_cols if c in shot_chart_df.columns]

    shot_chart_df = shot_chart_df[keep_shot_cols].dropna(
        subset=["FGA_FREQ", "FG_PCT"]
    )

    if "ZONE_FG_PCTL" not in shot_chart_df.columns:
        shot_chart_df["ZONE_FG_PCTL"] = np.nan

except Exception:
    shot_chart_df = pd.DataFrame(
        columns=["Zone", "FGA_FREQ", "FG_PCT", "ZONE_FG_PCTL"]
    )

## 11. Create One-Page PDF

In [375]:
season_type_file = SEASON_TYPE.replace(" ", "_")
player_file = PLAYER_DISPLAY_NAME.replace(" ", "_").replace(".", "")
pdf_path = OUTPUT_DIR / f"{player_file}_{SEASON}_{season_type_file}_player_profile.pdf"

with PdfPages(pdf_path) as pdf:
    fig = plt.figure(figsize=(8.5, 11))
    fig.patch.set_facecolor("#FFFFFF")

    header_ax = fig.add_axes([0.045, 0.91, 0.91, 0.065])
    header_ax.axis("off")
    header_ax.add_patch(Rectangle((0, 0), 1, 1, transform=header_ax.transAxes, color=PRIMARY))

    header_ax.text(0.025, 0.62, PLAYER_DISPLAY_NAME,
                   color="white", fontsize=20, fontweight="bold",
                   va="center", ha="left")

    header_ax.text(0.025, 0.24,
                   f"Player Development Profile | {SEASON} {SEASON_TYPE}",
                   color=SECONDARY, fontsize=8.5, fontweight="bold",
                   va="center", ha="left")

    if logo_img is not None:
        logo_ax = fig.add_axes([0.842, 0.914, 0.09, 0.058])
        logo_ax.axis("off")
        logo_ax.imshow(logo_img)
    else:
        header_ax.text(0.975, 0.50, TEAM_ABBR,
                       color="white", fontsize=22, fontweight="bold",
                       va="center", ha="right", alpha=0.95)

    fig.patches.extend([
        Rectangle((0.043, 0.575), 0.435, 0.315, transform=fig.transFigure,
                  facecolor="#F7FBFF", edgecolor="#D8E8F8", linewidth=0.9, zorder=-1),
        Rectangle((0.522, 0.575), 0.435, 0.315, transform=fig.transFigure,
                  facecolor="#FFF8F8", edgecolor="#F1D6D6", linewidth=0.9, zorder=-1),
        Rectangle((0.043, 0.320), 0.435, 0.215, transform=fig.transFigure,
                  facecolor="#FFFFFF", edgecolor="#D8DEE6", linewidth=0.9, zorder=-1),
        Rectangle((0.522, 0.320), 0.435, 0.215, transform=fig.transFigure,
                  facecolor="#FFFFFF", edgecolor="#D8DEE6", linewidth=0.9, zorder=-1),
        Rectangle((0.043, 0.090), 0.914, 0.195, transform=fig.transFigure,
                  facecolor="#FFFFFF", edgecolor="#D8DEE6", linewidth=0.9, zorder=-1),
    ])

    ax_snapshot = fig.add_axes([0.055, 0.605, 0.405, 0.275])
    draw_pdf_table(
        ax_snapshot,
        snapshot_pdf,
        "Player Snapshot",
        PRIMARY,
        PRIMARY,
        font_size=7.15,
        header_font_size=7.85,
        first_col_width=0.34,
        wrap_width=42,
        table_height=0.915,
    )

    ax_radar = fig.add_axes([0.605, 0.618, 0.29, 0.230], polar=True)
    draw_radar(ax_radar, radar_labels, radar_values, PRIMARY, SECONDARY)

    fig.text(0.535, 0.872, "PERCENTILE GAME SHAPE",
             fontsize=10.2, fontweight="bold", color=PRIMARY, ha="left")

    ax_shot_legend = fig.add_axes([0.055, 0.350, 0.090, 0.135])
    draw_shot_percentile_legend(ax_shot_legend, target_shot_group)

    ax_shot = fig.add_axes([0.160, 0.325, 0.315, 0.190])
    draw_shot_court(ax_shot, shot_chart_df, PRIMARY, SECONDARY) 

    ax_role = fig.add_axes([0.535, 0.330, 0.405, 0.195])
    draw_role_archetype_card(ax_role, role_archetype_data, PRIMARY, SECONDARY)

    ax_comps = fig.add_axes([0.055, 0.10, 0.875, 0.177])
    draw_pdf_table(
        ax_comps,
        comps_pdf.head(6),
        "Most Similar Player-Seasons",
        PRIMARY,
        PRIMARY,
        font_size=6.65,
        header_font_size=7.45,
        first_col_width=0.200,
        wrap_width=30,
        table_height=0.825,
        row_scale_y=1.20,
        note="PTS, REB, and AST are shown per 100 possessions.",
        note_y=0.000,
    )

    fig.text(0.055, 0.025,
             "Data: NBA Stats API / nba_api.\nRadar percentiles compare qualified player-seasons within inferred role group; shot zones compare current-season FG% within broad position group.",
             fontsize=6.5, color="#6B7280", ha="left")

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print(f"Saved PDF: {pdf_path}")

Saved PDF: player_development_reports/Andrew_Wiggins_2015-16_Regular_Season_player_profile.pdf
